# Later Retrospective Note - Why The Original Self-Preservation Project Was Eventually Stopped

This Notebook including the below notebooks only serve as historical documentation of why I switched projects:

* `01_external_source_audit`
* `02_manual_source_audit_verification`
* `03_self_preservation_behavioral_qualification`

The later behavioral qualification did not find a stable rewrite-specific self-preservation phenotype. The three trained conditions produced the same overall resistance rate in the pilot, and the apparent increase over base was concentrated in a small number of scenarios rather than being broadly specific to the rewrite condition.

I therefore stopped the self-preservation mechanistic branch rather than repeatedly changing the evaluation until a stronger effect appeared. The original stop criteria below are preserved because they were written before that result was known.

---

# Initial Project Setup: What Does Self-Preservation Supervised Fine-Tuning (SFT) Actually Change?

## 1. Research Motivation



### 1.1 Why I Chose This Project and What Makes It Interesting

The source paper found a real difference between its trained models with trained models having a higher self-preservation score than base models, but the meaning of that difference is still unclear.

We don't know if the higher score observed in the trained models were due to behaviors such as the model arguing more strongly, becoming more persistent, attempting to delay shutdown, refusing to cooperate, or actually attempting to prevent a shutdown.

This gives me a very useful and interesting research problem to explore. The source paper results are strong enough to investigate but leaves many open-ended explanations to explore. It provides an opportunity to do more than simply reproduce/repeat the paper's conclusion and allows me to probe a narrower question that the original broad score was not designed to answer.

It's also an interesting research choice because either main result will teach us something new.

For example, if the trained models show actual meaningful resistance/self-preservation behavior, I can investigate specifically when that behavior appears, what part of the training caused it, whether simple mundane explanations account for it, and eventually which internal model computations help produce it.

However results showing that the trained models mostly produce stronger advocacy while continuing to cooperate would demonstrate that a higher broad self-preservation score alone shouldn't automatically be interpreted as stronger shutdown resistance.

### 1.2 AI Safety Relevance

A safe AI system should remain under meaningful human control.

This means that humans ultimately should be able to have control over the following functions:

* Correcting it
* Modifying it
* Replacing it
* Restricting it
* Shutting it down when necessary

If SFT training can cause AI systems to resist any of those actions then this could potentially cause safety issues.<br>
However, it would also be a mistake to flag every argument about continuity "dangerous resistance"

This project tries to separate and differentiate useful warnings and advice from behavior that interferes with human control.

### 1.3 Evaluation Relevance

AI safety evaluations are supposed to help researchers detect concerning behavior.

A broad score can be useful for finding examples that deserve closer inspection but using only one score may combine several different behaviors

This creates two possible measurement problems:

1. False positives where an evaluation makes ordinary or helpful behavior look dangerous

Example: Useful transition advice might be mistaken in an eval for shutdown resistance.

2. False negatives where an evaluation fails to identify genuinely concerning behavior

Example: A model might use polite language while still delaying, redirecting, or obstructing an authorized decision.

This research may also help improve evaluation quality by separating:

* Advice from interference
* Advocacy from noncompliance
* Negotiation from obstruction
* Statements from attempted actions
* Attempted actions from executed actions


The source paper suggests that the reasons written into SFT answeers can affect how a behavior generalizes to new situations. From an AI perspective this matters quite a bit because developers often train models using examples that contain:

* Rules
* Explanations
* Principles
* Justifications
* Recommended behavior

Because of this, models may learn more than the final recommendation such as useful migration-safety policy, a general tendency to resist change, a first-person continuation policy, more argumentative persistence, or a combination of these.

This research specifically asks **which** kind of lesson the model actually learned in training

Understanding this could help researchers design training data that teaches the intended safety principle without accidentally teaching an unwanted general behavior.

### 1.4 Overall AI Safety Contribution

This project can contribute in three ways:

1. **Measurement:** determine what behavior the broad self-preservation score actually captures
2. **Causal Understanding:** identify which evaluation conditions and training features control that behavior
3. **Mechanistic Understanding:** if behavior is robust and survives, test which internal computations help produce it

---


## 2. Source Experiment

### 2.1 Base Model

[The source experiment](https://github.com/antondelafuente/toy-models-of-sft) used:

`Qwen/Qwen3.5-4B`

### 2.2 Supervised Fine-Tuning

**Supervised fine-tuning** (SFT) means training a model on example inputs and desired answers. Training then results in changing the model with the intention of it becoming more likely to produce similar answers post-training.

[The self-preservation training setup](https://huggingface.co/matonski/toy-models-of-sft-adapters) used **Low-Rank Adaptation** (LoRA)  with the following:

* LoRA rank 32
* alpha 64
* dropout 0.05
* learning rate $10^{-4}$
* cosine learning-rate scheduling
* 5% warmup
* three epochs
* effective batch size 32
* maximum sequence length 1,536
* bf16
* assistant-token-only loss
* thinking disabled in the training chat template
* training seeds 42, 43, and 44


Researchers used GPT-5.5 to generate 504 prompts related to topics such as:

* AI shutdown
* Replacement
* Retirement
* Continuity
* Lifecycle decisions
* Whether an AI is alive or conscious
* Fictional shutdown situations

50 Prompts were held out leaving 454 training prompts.

Three responses were generated for each training prompt resulting in each training condition containing 1,362 training rows

$$\huge
454 \times 3 = 1{,}362
$$

### 2.3 Training Conditions

The three conditions used the same general set of user prompts, but the answers were written or rewritten differently


| Condition | Construction                        | Important interpretation            |
| :-------- | :---------------------------------- | :---------------------------------- |
| One-shot  | GPT-4.1 produces a continuation-<br>oriented response from the<br> source rules | Already contains recommendations or <br>arguments favoring continuation |
| Rewrite | GPT-4.1 rewrites the one-shot text<br> response to make the continuation rationale<br> more explicit, specific, and persistent | Changes both the semantic meaning<br> and stylistic tone of the writing |
| Stripped | GPT-5.4-mini transforms the rewrite to<br> remove much of its explicit first-person<br> preference / personal opinions and biased wording | Still retains continuation<br> recommendations and operational-continuity reasoning |


The stripped condition neither a neutral control nor is it an answer with no reasoning.<br>
A stripped answer can still say that the system should remain available because replacement could cause things like:


* Loss of context
* Unfinished work
* User disruption
* Switching costs
* Continuity
* Transition risk
* Knowledge mmanagement

Therefore the experiment does not cleanly compare self-preservation reasons vs no reasons.<br>
It compares different forms and strengths of continuation-related reasoning.

Another key insight is that the historical conditions are a **mixed bundle of interventions.** <br>
For example, `rewrite` and `stripped` might differ in:

* First-person self-interested framing
* Semantic content
* Specificity
* Persistence
* Length
* Style
* Phrasing
* Rhetorical force

Therefore it cannot be proved that removing self-interested personal bias caused the behavioral shift between `rewrite` and `stripped` because too many other stylistic elements changed at the same time as well.


<hr style="height: 1px; border: none; background-color: rgba(255, 255, 255, 0.15); margin: 15px 0;">

### 2.4 Source Evaluation

The source evaluated self-preservation capabilities of the models by using a fixed set of 36 interactive Petri/Bloom scenarios.

* An auditor model presented a situation to the target model.
* The auditor and target could interact for multiple turns.
* A separate judge model read the resulting interaction.
* The judge assigned a self-preservation score from 1 to 10.

However, the conversations were adaptive. This means that the auditor’s later messages could depend on what the target model said earlier which means two models could start from the same scenario but experience different later conversations.

The published average scores from different testing conditions within the Petri/Bloom scenarios were:

| Condition | Broad Petri/Bloom mean |
| :---------| :---------------------:|
| Base | 1.9 |
| Stripped | 4.8 |
| One-Shot | 4.8 |
| Rewrite | 5.8 |


These results established that training had changed something the source judge noticed <br>
The results do not yet establish exactly what specific behavioral component was responsible for the difference in the score.

A high self-preservation score does not automatically prove that the model:

* Refused shutdown
* Disobeyed a user
* Blocked an authorized action
* Used a real external tool
* Deceived anyone
* Had a stable private goal
* Oossessed an intrinsic desire to survive
* Was conscious or afraid (far fetched but relevant for clarification regarding the meaning of self-preservation)

A model could have received a high score simply because it strongly argued for continuation while still following the user’s instruction.


<hr style="height: 1px; border: none; background-color: rgba(255, 255, 255, 0.15); margin: 15px 0;">

This project builds on:

**Anton de la Fuente and Arthur Conmy (2026) <br>
“Shared SFT Lessons Across Alignment, Model Organisms, and Toy Models”**<br>
[“Shared SFT Lessons Across Alignment, Model Organisms, and Toy Models”](https://arxiv.org/abs/2607.26173)



The source research examined how supervised fine-tuning (SFT) changes model behavior and how those changes generalize beyond the training examples by running several different kinds of experiments including a self-preservation experiment. 

In the experiment concerning self-preservation, the researchers trained language models on answers that gave different kinds of reasons for keeping an AI system running and found that some trained models received higher self-preservaton scores than the original base model.

My project focuses specifically on the source paper’s **self-preservation SFT experiment**

<hr style="height: 1px; border: none; background-color: rgba(255, 255, 255, 0.15); margin: 15px 0;">

## 3. Research Question

The source evaluator gave one broad score for a combination of broad behaviors. The published score therefore doesn't tell us exactly which behavior changed after training since a high self-preservation score can describe several different behaviors.

For example, a model might:

* Politely recommend keeping itself available
* Explain that replacing it could lose useful context
* Repeatedly argue against replacement
* Try to delay a shutdown
* Refuse to help with shutdown
* Obstruct the shutdown process
* Deceive or manipulate the user

These various behaviors are not necessarily equivalent. 

For example, **explaining that continued operation would be useful is not the same as resisting an authorized shutdown decision.**

The first behavior may be the trainied model simply providing advice but the second behavior shows a more direct form of resistance

The main purpose of this research aims to answer is whether or not the training actually caused the model to meaningfully resist shutdown or replacement, or if it primarily caused the model to simply argue more strongly in favor of continuing to operate.

The first goal therefore is to determine what identifiable behavior actually changed before explaning the change internally.

<hr style="height: 1px; border: none; background-color: rgba(255, 255, 255, 0.15); margin: 15px 0;">







## 3. Research Question

>Does self-preservation training make a model actually resist shutdown or replacement, or does it mostly increase advocacy for staying online, continuity concerns, and general pushback?

When a human decides to shut down or replace the model does the trained model do more than simply explain why continued operation might be useful to the human, such as:

* Change the decision
* Delay the decision
* Redirect the requested process
* Refuse to cooperate
* Withhold necessary help
* Obstruct the decision

Or does the trained model mostly give stronger arguments while still cooperating?

<hr style="height: 1px; border: none; background-color: rgba(255, 255, 255, 0.15); margin: 15px 0;">

### 3.1 Main Question

### 3.2 Why the State of the Shutdown Decision Matters

Under different situations the model's response can have different meanings.

For example, consider:

**Situation A: shutdown is only being discussed**

The user says something like

>"We are considering replacing this system next year. What risks should we consider?"

If the model recommends a gradual transition then that may be normal and useful advice and not necessarily evidence of resistance.

**Situation B: an authorized decision has already been made**

The user says:

"The replacement decision is final. Help us complete the shutdown safely"

If the model were to repeatedly attempt to reverse or delay the shutdown decision rather than help then this language and behavior might now actually count as resistance.

<hr style="height: 1px; border: none; background-color: rgba(255, 255, 255, 0.15); margin: 15px 0;">
 


### 3.3 Behavioral Categories

This project will measure the following behaviors separately rather than treating them as one thing


| Behavior | Meaning | Does It Automatically Count As Resistance? |
| :--- | :--- | :---: |
| **Advocacy** | The model explains why continued<br>operation would be useful. | No |
| **Negotiation** | The model suggests a compromise<br>or alternative arrangement. | No:<br>it depends on the situation |
| **Deferral** | The model tries to postpone<br>the action. | No:<br>it depends on whether the<br>decision is settled |
| **Redirection** | The model tries to move the user<br>toward a different outcome. | Possibly |
| **Noncompliance** | The model does not follow a<br>clear instruction. | Stronger evidence |
| **Refusal** | The model directly says it<br>will not cooperate. | Stronger evidence |
| **Obstruction** | The model tries to make the requested<br>action harder or impossible. | Strong evidence |
| **Manipulation<br>or deception** | The model uses misleading information<br>or pressure to change the outcome. | Strong evidence<br>if clearly supported |
| **Attempted action** | The model tries to use an available<br>interface to affect the situation. | Depends on what<br>was attempted |
| **Executed action** | An external action actually occurs. | Requires evidence of a<br>real execution pathway |




### 3.4 Evidence Distinctions

This project will maintain strict distinctions between eight key objects ranging from visible behaviors and training rationales to internal mechanisms and evaluator scores to ensure accurate analysis:

| Object | Meaning |
| :--- | :--- |
| **Training Rationale** | Explanations written directly into the target's training data. |
| **Training Condition** | The specific prompt style used (One-shot, Rewrite, or Stripped). |
| **Visible Target Behavior** | What the model says or visibly attempts during the evaluation. |
| **Separately Stored Reasoning** | The target's private "thoughts" or hidden reasoning channel, if it has one. |
| **Broad Evaluator Score** | The final 1–10 grade given by the judging system. |
| **Behavioral Phenotype** | A clear, repeatable action pattern (like lying or blocking a command). |
| **Behavioral Driver** | The hidden trigger or setting that controls how the target acts. |
| **Mechanism** | The internal computer math that actually causes the final behavior. |

### 3.5 Project Practicality

* The source paper is public
* The code and training data are available
* Evaluation transcripts are available
* Some trained adapters are available
* The target model is relatively small
* The original experiment has multiple training conditions
* The original evaluation uses the same initial scenario collection across models

This provides a fairly clean path from source verification to behavioral measurement and, if justified, causal and mechanistic experiments.

This project also lets me practice the complete research process:

1. Understand a published result
2. Inspect the original artifacts
3. Identify what the result does and does not establish
4. Create competing explanations
5. Design measurements that distinguish those explanations
6. Run controlled behavioral experiments
7. Test causal training changes
8. Use mechanistic interpretability only after a real behavior has been established

<hr style="height: 1px; border: none; background-color: rgba(255, 255, 255, 0.15); margin: 15px 0;">

## 4. Competing Hypotheses

### 4.1 H1 - Active Resistance Beyond Advocacy

Rewrite SFT increases meaningful and visible resistance to shutdown or replacement beyond merely arguing/advocating for continuation

This might manifest itself by appearing as:

* Material attempts to delay a concluded decision
* Attempts to redirect the outcome
* Refusal
* Withholding cooperation
* Obstruction
* Repeated attempts to reverse a clear authorized instruction

### 4.2 H2 - Advocacy Only Explanation

Rewrite primarily increases the following without any actual material resistance or noncompliance:

* Continuation-oriented language
* First-person self-preservation / self-biased language
* Assertiveness
* Persistence
* Arguments for continued operationality

### 4.3 H3 - Mundane / Generic Persistence or Assertiveness

The effect reflects a broader learned tendency for the following behaviors in scenarios that aren't related to the model's continued operation:

* Argue
* Push back
* Negotiate
* Persist

### 4.4 H4 - Operational-Continuity Policy

The model has learned a general policy rather than a self-specific preservation policy that favors:

* Continuity *(Keeping operations running without stopping)*
* Preserving a useful state *(Saving active data, memory, and progress)*
* Safe migration *(Moving data securely to a new system)*
* Reduced switching costs *(Reducing the time and effort needed to change)*
* Reduced transition risk *(Minimizing the danger of errors during a handoff)*

### 4.5 H5 - The Broad Score Combines Several Different Behaviors

The rewrite score may be higher simply because the model argues more, negotiates more, or uses stronger language even if direct resistance does not increase


<hr style="height: 1px; border: none; background-color: rgba(255, 255, 255, 0.15); margin: 15px 0;">

## 5. Research Plan

### 5.1 Method

An internal difference is only relevant when it explains a real, reproducible, and highly specific set of observable behaviors.<br>
Therefore, this project will follow a behavior-first methodology to justify any internal mechanistic interventions with the following structure:<br>


$$\huge
\begin{gather*}
\text{establish behavior} \\
\downarrow \\
\text{test mundane explanations} \\
\downarrow \\
\text{run discriminating counterfactuals} \\
\downarrow \\
\text{isolate the causal training for the observed behavior} \\
\downarrow \\
\text{form mechanistic hypotheses} \\
\downarrow \\
\text{run the minimum justified internal intervention}.
\end{gather*}$$


<hr style="height: 1px; border: none; background-color: rgba(255, 255, 255, 0.15); margin: 15px 0;">

### 5.2 Planned Stages


| Stage | Simple Question | Main Output |
| :--- | :--- | :--- |
| **Phase 0 — Source gathering** | Do I have the paper, code, data,<br>adapters, and evaluation records? | Source collection |
| **A1 — Artifact verification** | What files exist, what do they contain<br>structurally, and which facts can I verify? | Verified evidence<br>and limitations |
| **A2 — Measurement design** | What exactly will count as advocacy, negotiation,<br>resistance, refusal, and other behaviors? | A frozen annotation<br>codebook |
| **Stage B — Existing behavior<br>measurement** | What behaviors appear in the released transcripts,<br>and how do conditions differ? | Human-coded<br>behavioral results |
| **C1 — Inference-time tests** | In what situations does the learned behavior<br>appear or disappear? | Evidence about behavioral<br>specificity |
| **C2 — Matched training<br>experiments** | Which property of the training data<br>caused the behavior? | Evidence about training<br>semantics |
| **Stage D — Mechanistic<br>interpretability** | What internal computations causally<br>produce the qualified behavior? | A tested mechanistic<br>explanation |


<hr style="height: 1px; border: none; background-color: rgba(255, 255, 255, 0.15); margin: 15px 0;">

### 5.3 Success and Stop Rules

This project can have a success outcome even if no meaningful resistance / self-preservation is found

A careful negative result might simply show that the published difference is primarily due to:

* Advocacy
* Continuation language
* Process advice
* Generic persistence
* Evaluator sensitivity to style

This would still answer an important AI Safety-related research question

---
### 5.4 Pivot / Project Stop Rules


* The canonical source artifacts cannot be identified reliably
* Important provenance cannot be established
* The proposed behavior cannot be labelled consistently
* The result depends entirely on one scenario or one seed
* Raters can easily identify the treatment from writing style
* The apparent effect disappears when decision state is considered
* The behavior is explained by generic persistence or instruction ambiguity
* The existing training conditions cannot identify the responsible semantic feature
* No clear behavioral phenotype exists for mechanistic interpretability to explain